In [2]:
sentence = "the capital of united states and the capital of france"
context_length = 32

In [3]:
import torch
from my_tokenizer import Tokenizer

tokenizer = Tokenizer("tokenizer.json")

torch.manual_seed(1)

embeddings = torch.nn.Embedding(num_embeddings=64, embedding_dim=4)

In [4]:
sentence = "the capital of united states and the capital of france"
tokens = tokenizer.encode(sentence)
tokens = torch.tensor(tokens)

meanings = embeddings(tokens)

C:\Users\Zeynep\AppData\Local\Temp\ipykernel_12636\3387454732.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tokens = torch.tensor(tokens)


In [5]:
def get_position_encoding(context_length, embedding_dim, base=10000, device="cpu"):
    """
    Generate sinusoidal position encoding.

    Args:
        context_length (int): Length of the context (sequence length).
        embedding_dim (int): Dimension of the embeddings.

    Returns:
        list: A list of lists representing the position encoding matrix.
    """
    import math
    import torch
    
    p_embeddings = torch.zeros((context_length, embedding_dim), device=device)

    for pos in range(context_length):
        for i in range(embedding_dim // 2):
            p_embeddings[pos, 2 * i] = math.sin(pos / (base ** (2 * i / embedding_dim)))
            if i+1 < embedding_dim :
                p_embeddings[pos, 2 * i + 1] = math.cos(pos / (base ** (2 * i / embedding_dim)))
    return p_embeddings.unsqueeze(0)


In [6]:
pos_embeddings = get_position_encoding(20, embedding_dim=4)
pos_embeddings

tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998],
         [ 0.1411, -0.9900,  0.0300,  0.9996],
         [-0.7568, -0.6536,  0.0400,  0.9992],
         [-0.9589,  0.2837,  0.0500,  0.9988],
         [-0.2794,  0.9602,  0.0600,  0.9982],
         [ 0.6570,  0.7539,  0.0699,  0.9976],
         [ 0.9894, -0.1455,  0.0799,  0.9968],
         [ 0.4121, -0.9111,  0.0899,  0.9960],
         [-0.5440, -0.8391,  0.0998,  0.9950],
         [-1.0000,  0.0044,  0.1098,  0.9940],
         [-0.5366,  0.8439,  0.1197,  0.9928],
         [ 0.4202,  0.9074,  0.1296,  0.9916],
         [ 0.9906,  0.1367,  0.1395,  0.9902],
         [ 0.6503, -0.7597,  0.1494,  0.9888],
         [-0.2879, -0.9577,  0.1593,  0.9872],
         [-0.9614, -0.2752,  0.1692,  0.9856],
         [-0.7510,  0.6603,  0.1790,  0.9838],
         [ 0.1499,  0.9887,  0.1889,  0.9820]]])

In [7]:
meanings_in_sentence = meanings + pos_embeddings
meanings_in_sentence

tensor([[[-1.5256,  0.2498, -0.6540, -0.6095],
         [ 0.0966,  0.3381, -0.2197,  1.0013],
         [ 0.8091, -1.0253, -0.9598, -0.6093],
         [-0.6037, -1.1921, -0.1997,  1.0009],
         [-1.4689, -0.3499, -0.7373,  0.7477],
         [-1.7038,  0.0815, -0.1797,  1.0001],
         [-0.5017,  2.6473,  0.2884,  1.4658],
         [-0.0879,  0.5517, -0.1597,  0.9989],
         [ 0.2924, -1.3063,  0.7795,  1.1959],
         [ 0.3323, -0.5694,  1.0387, -0.3880],
         [-1.2889, -1.0412, -0.1298,  0.9963],
         [-1.1110,  0.2972, -0.0481,  0.9652],
         [-1.2814,  0.6417, -0.1100,  0.9941],
         [-1.1054,  0.1572, -0.5243, -0.6179],
         [ 0.2458, -0.0654, -0.0901,  0.9915],
         [ 0.5501, -1.3689, -0.8303, -0.6203],
         [-1.0327, -1.1598, -0.0703,  0.9886],
         [-1.6735,  0.0286, -0.6081,  0.7341],
         [-1.4958,  0.4582, -0.0506,  0.9852],
         [ 2.0292,  0.9166,  0.3466,  0.2085]]], grad_fn=<AddBackward0>)

In [8]:
import plotly.graph_objects as go
import plotly.offline

def plot_dots(sentences_data, title, dims=[0, 1, 2]):
  data = [
    go.Scatter3d(
      x=sentence_data["words"][:, dims[0]],
      y=sentence_data["words"][:, dims[1]],
      z=sentence_data["words"][:, dims[2]],
      mode="markers+text",
      marker=dict(
        size=6,
        color=sentence_data["color"],
      ),
      text=sentence_data["labels"],
      hoverinfo="text",
    ) for sentence_data in sentences_data
  ]

  layout = go.Layout(
    scene=dict(
      xaxis_title="Sertlik",
      yaxis_title="Parlaklık",
      zaxis_title="Kırmızılık",
    ),
    title=title,
  )

  fig = go.Figure(data=data, layout=layout)
  plotly.offline.iplot(fig)

In [9]:
sentences = [
    {
        "words": meanings_in_sentence[0].detach().numpy(),
        "labels": tokenizer.tokenize(sentence),
        "color": "red",
    },
    {
        "words":meanings.detach().numpy(),
        "labels": tokenizer.tokenize(sentence),
        "color": "blue",
    }
    ]

plot_dots(sentences, "Sinusoidal Positional Encoding Uygulaması")